
# 🥉 Bronze Layer - Raw Ingestion
**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Goal:** Read the raw CSV and register it as a Delta Table with zero transformations.  
This layer is the single source of truth - nothing is modified here.

In [0]:
# Verify that the Spark session and Python environment are ready
# This is the first check we run before any data processing
import sys

try:
    # spark.version accesses the Spark session object that Databricks
    # creates automatically — no need to initialize it manually
    spark_version = spark.version
    
    # sys.version returns the full Python version string
    # we split by space and take the first element to get just the number
    # Example: "3.12.3 (main, ...)" → "3.12.3"
    python_version = sys.version.split(" ")[0]
    
    # Print a clean summary of the environment
    print("=" * 45)
    print("  ENVIRONMENT CHECK")
    print("=" * 45)
    print(f"  Spark version  : {spark_version}")
    print(f"  Python version : {python_version}")
    print("=" * 45)
    print("  Status: cluster is active ✓")
    print("=" * 45)

except Exception as e:
    # If anything above fails, we print a clear error message
    # and re-raise the exception so the notebook stops execution
    # We never want to silently ignore errors in a data pipeline
    print(f"  ✗ Cluster check failed: {e}")
    raise

In [0]:
# Read data directly from the table we created via Data Ingestion
# spark.table() reads from the Databricks catalog using the full table path:
# "catalog.schema.table_name" → "main.default.warehouse_and_retail_sales"
# inferSchema is not needed here because the table already exists in the catalog

df_bronze = spark.table("main.default.warehouse_and_retail_sales")

# Count total rows and columns to verify the data loaded correctly
# f-string with :, formats the number with comma separators → 307,548
print(f"Rows   : {df_bronze.count():,}")
print(f"Columns: {len(df_bronze.columns)}")

# printSchema() shows the data type of each column
# In Bronze everything should be string — we intentionally keep raw types
print("\nSchema:")
df_bronze.printSchema()

## Bronze Layer — Schema Overview

The raw dataset contains **307,645 rows** and **9 columns**.

Databricks automatically inferred the data types during upload:

| Column | Type | Description |
|--------|------|-------------|
| YEAR | long | Year of the transaction (2020–2026) |
| MONTH | long | Month number (1–12) |
| SUPPLIER | string | Distributor name |
| ITEM CODE | string | Product identifier |
| ITEM DESCRIPTION | string | Full product name |
| ITEM TYPE | string | Category: WINE, BEER or LIQUOR |
| RETAIL SALES | double | Units sold at retail level |
| RETAIL TRANSFERS | double | Units transferred between stores |
| WAREHOUSE SALES | double | Units sold from warehouse |

> **Note:** All columns have `nullable = true`, meaning null values
> may exist in any column. This will be handled in the Silver layer.

> **Note:** Since Databricks inferred numeric types correctly on upload,
> the Silver layer will focus on: creating a unified `date` column,
> standardizing text fields, and computing `total_sales`.

In [0]:
# Preview the first 10 rows
# display() is a Databricks-native function — better than .show()
# It renders an interactive table instead of plain text
display(df_bronze.limit(10))

In [0]:
# Check null values per column
# In Bronze we only observe — we never fix anything here
# If nulls exist, they will be handled in the Silver layer
from pyspark.sql import functions as F

null_counts = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_bronze.columns
])

print("Null values per column:")
display(null_counts)

## Bronze Layer — Data Quality Observations

| Column | Null Count | Action in Silver |
|--------|-----------|-----------------|
| YEAR | 0 | No action needed |
| MONTH | 0 | No action needed |
| SUPPLIER | 0 | Standardize to uppercase |
| ITEM CODE | 167 | Keep as-is — investigate in Silver |
| ITEM DESCRIPTION | 0 | Trim whitespace |
| ITEM TYPE | 1 | Fill with "UNKNOWN" |
| RETAIL SALES | 3 | Fill with 0.0 |
| RETAIL TRANSFERS | 0 | No action needed |
| WAREHOUSE SALES | 0 | No action needed |

> **Bronze rule:** We observe and document — we never modify data in this layer.

In [0]:
# Save as official Bronze Delta Table
# This is our controlled Bronze layer — separate from the raw upload
# mode="overwrite" ensures a clean write every time this notebook runs
# saveAsTable() registers it in the Databricks catalog for easy access

try:
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("delta.columnMapping.mode", "name") \
        .saveAsTable("main.default.bronze_warehouse_sales")

    # Verify the table was saved correctly
    row_count = spark.table("main.default.bronze_warehouse_sales").count()

    print("=" * 45)
    print("  BRONZE LAYER — COMPLETE")
    print("=" * 45)
    print(f"  Table : main.default.bronze_warehouse_sales")
    print(f"  Rows  : {row_count:,}")
    print(f"  Status: saved successfully ✓")
    print("=" * 45)

except Exception as e:
    print(f"  ✗ Failed to save Bronze table: {e}")
    raise

## Bronze Layer — Summary

The Bronze layer is now complete. The raw dataset has been ingested and 
registered as a Delta Table with zero transformations applied.

### What was done
- Read 307,645 rows from the raw CSV uploaded via Databricks Data Ingestion
- Validated schema — all 9 columns confirmed
- Identified null values across all columns
- Saved as Delta Table: `main.default.bronze_warehouse_sales`

### Key findings
- **ITEM CODE:** 167 null values — products without an identifier
- **ITEM TYPE:** 1 null value — missing category
- **RETAIL SALES:** 3 null values — missing sales figures

### Next step
All null values and data quality issues will be handled in the 
**Silver layer** (`02_silver_transformation`), where data will be 
cleaned, typed correctly, and enriched with new columns.